# Workbench do Solver

Notebook de demonstracao orientado por narrativa. O objetivo aqui e responder, em ordem, qual problema esta sendo resolvido, o que torna o cenario dificil, como a rede deve ser lida, qual foi a solucao produzida e qual takeaway deve entrar na apresentacao.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

PROJECT_ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'pyproject.toml').exists())
NOTEBOOK_DIR = PROJECT_ROOT / 'notebook'
import sys
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from solver_workbench_support import (
    analyze_scenario,
    build_takeaway,
    compile_scenario,
    export_presentation_bundle,
    load_scenario_artifacts,
    plot_base_graph,
    plot_kpi_dashboard,
    plot_solution_graph,
    route_sequences,
    run_scenario,
    summarize_dataset,
    summarize_orchestration,
)

SCENARIO = 'operacao_controlada'
MAX_ITERATIONS = 50
SEED = 1
WITH_BASEMAP = False
EXPORT_DIR = PROJECT_ROOT / 'docs' / 'apresentacao' / 'assets' / 'generated' / 'notebook'
display(Markdown(f'**Configuracao ativa**: cenario=`{SCENARIO}`, max_iterations=`{MAX_ITERATIONS}`, seed=`{SEED}`'))

## 1. Qual problema estou resolvendo?

In [ ]:
compile_scenario(SCENARIO)
ARTIFACTS = load_scenario_artifacts(SCENARIO)
DATASET_SUMMARY = summarize_dataset(ARTIFACTS)
SCENARIO_ANALYSIS = analyze_scenario(ARTIFACTS)
DATASET_SUMMARY | {'dominant_bottleneck': SCENARIO_ANALYSIS['dominant_bottleneck'], 'priority_ratio': SCENARIO_ANALYSIS['priority_ratio']}

## 2. O que torna esse cenario dificil?

In [ ]:
display(Markdown(
    f"**Leitura do gargalo**: `{SCENARIO_ANALYSIS['dominant_bottleneck']}`. "
    f"Janela media: `{SCENARIO_ANALYSIS['avg_window_hours']}` h, "
    f"pressao financeira: `{SCENARIO_ANALYSIS['cash_pressure_ratio']}`, "
    f"pressao volumetrica: `{SCENARIO_ANALYSIS['volume_pressure_ratio']}`."
))
SCENARIO_ANALYSIS

## 3. Como a rede-base deve ser lida?

In [ ]:
plot_base_graph(ARTIFACTS, with_basemap=WITH_BASEMAP)

## 4. Quais parametros de execucao governam a leitura da solucao?

In [ ]:
{
    'scenario': SCENARIO,
    'max_iterations': MAX_ITERATIONS,
    'seed': SEED,
    'materialize_snapshot': True,
    'service_policy': 'maximize_attendance_v1',
}

## 5. O que o solver construiu?

In [ ]:
ORCHESTRATION = run_scenario(
    SCENARIO,
    max_iterations=MAX_ITERATIONS,
    seed=SEED,
    materialize_snapshot=True,
)
SUMMARY = summarize_orchestration(ORCHESTRATION)
SUMMARY

In [ ]:
plot_solution_graph(ORCHESTRATION, ARTIFACTS, with_basemap=WITH_BASEMAP)

## 6. Qual foi o saldo operacional em KPI e sequencia?

In [ ]:
plot_kpi_dashboard(ORCHESTRATION)

In [ ]:
route_sequences(ORCHESTRATION)

## 7. Qual takeaway entra na apresentacao?

In [ ]:
display(Markdown(build_takeaway(ORCHESTRATION, ARTIFACTS)))
export_presentation_bundle(ORCHESTRATION, ARTIFACTS, output_dir=EXPORT_DIR, with_basemap=WITH_BASEMAP)